In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Data preparation

In [120]:
df = pd.read_csv('./lead-scoring-dataset-2026.csv')

In [121]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [122]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [123]:
categorical = list(df.dtypes[df.dtypes == 'str'].index)
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
target = 'converted'
features = categorical + numerical

In [124]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [125]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [126]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Validation framework

In [127]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [128]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [129]:
print(f"# rows: {len(df)}")
print(f"# train rows: {len(df_train)}")
print(f"# val rows: {len(df_val)}")
print(f"# test rows: {len(df_test)}")

# rows: 5000
# train rows: 3000
# val rows: 1000
# test rows: 1000


In [130]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

In [131]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
1,events,finance,employed,south_america,71662.0,0,0,0.09,0
2,referral,NA,employed,asia,40220.0,2,5,0.55,1
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
4,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [132]:
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

In [133]:
del df_train[target]
del df_val[target]
del df_test[target]

### Q.1
What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology` $\checkmark$
- `healthcare`
- `retail`

In [134]:
df_full_train['industry'].value_counts()

industry
technology       918
retail           735
healthcare       665
finance          582
education        518
manufacturing    388
NA               194
Name: count, dtype: int64

### Q.2 
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score` $\checkmark$
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

In [135]:
corr_matrix = df_full_train[numerical].corr()
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.163680,0.122528,0.222493
number_of_courses_viewed,0.163680,1.000000,0.723040,0.757178
interaction_count,0.122528,0.723040,1.000000,0.916626
lead_score,0.222493,0.757178,0.916626,1.000000


### Q.3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source` $\checkmark$
- `employment_status`

In [136]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train[target])

mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).to_frame(name='mutual info score').round(2)

,mutual info score
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


### Q.4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65 $\checkmark$
- 0.75
- 0.85

In [137]:
def calculate_accuracy(features, c=1.0):
    train_dicts = df_train[features].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dicts = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    accuracy = (converted_decision == y_val).mean()
    return accuracy

In [138]:
accuracy = calculate_accuracy(features)

In [139]:
accuracy.round(2)

np.float64(0.64)

### Q.5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'` $\checkmark$
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [140]:
diffs = {}

for feature in features:
    new_features = features.copy()
    new_features.remove(feature)
    new_accuracy = calculate_accuracy(new_features)

    diff = np.abs(accuracy - new_accuracy)

    diffs[feature] = diff

for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    diff = diffs[feature]
    print(f"feature: '{feature}'\n difference: {diff}")
    print()

feature: 'lead_source'
 difference: 0.0030000000000000027

feature: 'number_of_courses_viewed'
 difference: 0.0020000000000000018

feature: 'interaction_count'
 difference: 0.04400000000000004



The feature-elimination results suggest that **`interaction_count` is the most influential feature**: removing it changed model accuracy by 0.044, compared with only 0.003 for `lead_source` and 0.002 for `number_of_courses_viewed`. This indicates that the model relies much more on `interaction_count`, while the other two features have relatively little effect on accuracy in this test.

### Q.6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001 $\checkmark$

> **Note**: If there are multiple options, select the smallest `C`.


In [145]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(f"C:{C:.6f}, accuracy:{calculate_accuracy(features, c=C)}")

C:0.000001, accuracy:0.598
C:0.000010, accuracy:0.598
C:0.000100, accuracy:0.613
C:0.001000, accuracy:0.645
